In [1]:
import pandas as pd
import numpy as np
import warnings
# warnings.filterwarnings('ignore')

In [2]:
ewb_plus_attendance = pd.read_csv('attendance-per-course-419983907987914818-2025-04-15-836425175229743104.csv')
ewb_plus_attendance.to_excel('text_excel.xlsx',index=False)

In [52]:
ewb_plus_attendance.to_excel('test.xlsx')

In [50]:
ewb_plus_melt  =    pd.melt(frame=ewb_plus_attendance,
                            id_vars=ewb_plus_attendance.columns[:25],
                            value_vars=ewb_plus_attendance.columns[26:]
                            ).sort_values(by='documentId')


# Extraer el índice y tipo de variable (date o status)
ewb_plus_melt['index'] = ewb_plus_melt['variable'].str.extract(r'attendance\\\.(\d+)\\\.')[0].fillna('-')
ewb_plus_melt['type'] = ewb_plus_melt['variable'].str.extract(r'(date|status)')[0].fillna('-')

ewb_plus_melt  = ewb_plus_melt[(ewb_plus_melt['type'] != "-") & (ewb_plus_melt['documentId'] != "undefined")]
filtro = ewb_plus_melt['variable'].str.contains('status|isoString', regex=True)
ewb_plus_melt = ewb_plus_melt[filtro]


pivoted = pd.pivot_table(
    ewb_plus_melt,
    index=['firstName', 'documentId', 'index'],
    columns='type',
    values='value',
    aggfunc='first'
).reset_index()

pivoted = pivoted[pivoted['date'] != 'undefined']

pivoted['datetime'] = pd.to_datetime(pivoted['date']).dt.strftime('%Y-%m-%d %H:%M')

# Pivot final con datetime como columnas
pivot_final = pivoted.pivot_table(
    index=['firstName', 'documentId'],
    columns='datetime',
    values='status',
    aggfunc='first'
).reset_index().fillna('-')

pivot_final['total_attendance'] = pivot_final[list(pivot_final.columns[2:])].apply(lambda row: (row == 'ATTENDED').sum(), axis=1)


# ewb_plus_melt.to_excel('test.xlsx')


In [53]:
pivot_final.to_excel('attendance_tracking.xlsx')

In [20]:
import pandas as pd

# Datos de ejemplo
data = [
    ['Adanolis', 'attendance.0.lesson.date.isoString.attendance.2.lesson.date.isoString', '2025-04-12T15:00:00Z'],
    ['Adanolis', 'attendance.0.status', 'ATTENDED'],
    ['Adanolis', 'attendance.1.lesson.date.isoString', '2025-04-12T13:00:00Z'],
    ['Adanolis', 'attendance.1.status', 'ATTENDED'],
    ['Adanolis', 'attendance.2.lesson.date.isoString', '2025-04-05T15:00:00Z'],
    ['Adanolis', 'attendance.2.status', 'NOT_ATTENDED'],
    ['Adanolis', 'attendance.3.lesson.date.isoString', '2025-04-05T13:10:00Z'],
    ['Adanolis', 'attendance.3.status', 'NOT_ATTENDED']
]

df = pd.DataFrame(data, columns=['firstName', 'variable', 'value'])

# Extraer el índice y tipo de variable (date o status)
df['index'] = df['variable'].str.extract(r'attendance\.(\d+)\.')[0]
df['type'] = df['variable'].str.extract(r'(date|status)')[0]

# Pivot intermedio para tener columnas date y status
pivoted = pd.pivot_table(
    df,
    index=['firstName', 'index'],
    columns='type',
    values='value',
    aggfunc='first'
).reset_index()

# Usar fecha completa (con hora) como columna
pivoted['datetime'] = pd.to_datetime(pivoted['date']).dt.strftime('%Y-%m-%d %H:%M')

# Pivot final con datetime como columnas
pivot_final = pivoted.pivot_table(
    index='firstName',
    columns='datetime',
    values='status',
    aggfunc='first'
).reset_index()


# Ordenar columnas cronológicamente
ordered_cols = ['firstName'] + sorted(pivot_final.columns[1:], key=lambda x: pd.to_datetime(x))
pivot_final = pivot_final[ordered_cols]

pivot_final['attended_count'] = pivot_final[list(pivot_final.columns[2:])].apply(lambda row: (row == 'ATTENDED').sum(), axis=1)



# Mostrar resultado
pivot_final


datetime,firstName,2025-04-05 13:10,2025-04-05 15:00,2025-04-12 13:00,2025-04-12 15:00,attended_count
0,Adanolis,NOT_ATTENDED,NOT_ATTENDED,ATTENDED,ATTENDED,2


In [22]:
df

,firstName,variable,value,index,type
0,Adanolis,attendance.0.lesson.date.isoString.attendance....,2025-04-12T15:00:00Z,0,date
1,Adanolis,attendance.0.status,ATTENDED,0,status
2,Adanolis,attendance.1.lesson.date.isoString,2025-04-12T13:00:00Z,1,date
3,Adanolis,attendance.1.status,ATTENDED,1,status
4,Adanolis,attendance.2.lesson.date.isoString,2025-04-05T15:00:00Z,2,date
5,Adanolis,attendance.2.status,NOT_ATTENDED,2,status
6,Adanolis,attendance.3.lesson.date.isoString,2025-04-05T13:10:00Z,3,date
7,Adanolis,attendance.3.status,NOT_ATTENDED,3,status
